# M6.A6 — 초기 모델 선정 + 평가 지표 확정

> 산출 근거: `docs/plan/ai/phase_06_model.md` M6.A6 · 선행: `04_baselines.ipynb`(비교·승자 V1)
> 하네스: M6.A4 확정 규칙(`../data_prep/preprocess.py`) · 작성: 2026-07-27

🔒 **공개 저장소 데이터 정책** — 실매장 매출의 절대 금액은 커밋하지 않는다. 본 노트북은 출력 제거
상태로 추적되며, 본문은 비율·배수·sMAPE(%)·하이퍼파라미터만 사용한다. 전체 수치는 로컬 재실행으로 확인.

**목적** — M6.A5 승자(V1)를 초기 모델로 확정하고, 경량 하이퍼파라미터 튜닝(선택 fold 한정)과
최종 "모델 카드"를 산출한다. 결과는 `model_spec.md` §3·§7에 반영(docs PR).

**사전 등록 원칙**
- 튜닝은 **선택 fold 5개 평균 MAE**만 최적화. **test(2026-04)는 재개봉하지 않는다** —
  공식 test 성적은 04 노트북의 스크리닝 V1 결과(sMAPE 30.0%, MA-7 대비 -19.6%)가 대표.
- 튜닝 채택 기준(탐색 전 고정): 평균 MAE **-2% 이상 개선 + fold 과반(3/5) 개선** — 미달 시 단순성
  우선으로 스크리닝 설정 유지.

## 판정 요약 (TL;DR)

1. **초기 모델 확정** — 주 모델 **LightGBM 비율 타깃 하이브리드(V1)**: 타깃 `log1p(y)−log1p(roll7_mean)`,
   복원 `ŷ = expm1(pred + log1p(roll7))`, keep 20열, 워밍업 NaN 라벨 명시 제거. 보조 baseline **MA-7**
   (fallback + drift 감시). 근거: 04 노트북 — 12개 후보 중 유일하게 MA-7을 유의미하게 이김.
2. **튜닝 채택(V1-t)** — Optuna 60 trials(TPE, seed 42, 선택 fold만): **-2.6% · 5개 fold 중 3개 개선**으로
   사전 기준 충족. 채택 파라미터는 스크리닝보다 보수적(leaves 15→9, lr 0.05→0.026, subsample·colsample 축소)
   — 소표본(선택 train 112~240일)에 정합적 방향. 선택-과적합 여지는 인정하며 운영 재학습 모니터링
   (MA-7 대비 skill)으로 사후 검증한다.
3. **평가 지표 확정** — **MAE(주 지표·선정 기준) + sMAPE(보고용)**. 구 후보(MAE+MAPE+R²)에서 **MAPE 제외
   정정**(주문 1건 소액일이 분모 왜곡 — EDA §2.4). 운영 목표(상대 기준): **naive-요일 대비 skill ≥ +15%,
   MA-7 우위 유지** — MA-7에 지기 시작하면 drift 경보(ml_pipeline §10 연계).
4. **타깃 명확화 (담당자 확인 포인트)** — 현 초기 모델의 타깃은 **매장 일 매출(1일 선행)**. spec의
   "메뉴별 1~3일 수요"와의 간극은 Phase 7에서 분해 설계(카테고리·메뉴 비중 규칙, 2~3일 선행 확장)로
   별도 확정 필요 — EDA §3 근거(메뉴 레벨 표본 부족: 재개장 후 신규 메뉴 이력 49일 이하).
5. **spec 반영(docs PR)** — model_spec §2(비교 완료 주석)·§3(초기 모델·라이브러리)·§5(확정 피처 참조)·
   §7(CV 구체화·평가 지표)·말미 미확정 목록 정리.

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

warnings.filterwarnings("ignore")

_here = Path.cwd()
AI_DIR = next(p for p in [_here.parent, _here, _here / "AI"] if (p / "data_prep").exists())
sys.path.insert(0, str(AI_DIR / "data_prep"))
import preprocess as pp

PAL = {"blue": "#2a78d6", "orange": "#eb6834", "ink2": "#52514e"}
_installed = {f.name for f in fm.fontManager.ttflist}
plt.rcParams.update({
    "font.family": [f for f in ("AppleGothic", "Apple SD Gothic Neo", "NanumGothic") if f in _installed] or ["sans-serif"],
    "axes.unicode_minus": False, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "figure.dpi": 100,
})

feat = pd.read_parquet(AI_DIR / "data/processed/features_daily.parquet").set_index("date")
feat, _ = pp.impute_weather(feat)
ob = feat[feat.is_open].copy()
y = ob.total_amount
r7 = ob.roll7_mean
folds = pp.make_monthly_folds(ob.index)
SEL, TEST = folds[:-1], folds[-1]
print("선택 fold:", [f["month"] for f in SEL], "| test(봉인 — 본 노트북에서 미사용):", TEST["month"])

KEEP = ["is_holiday", "semester_week", "is_semester_first2w", "temp_avg", "temp_range",
        "lag1_sales", "lag1_tx", "lag_dow_sales", "roll7_mean", "roll4dow_mean", "roll7_atv",
        "is_post_renewal", "days_since_reopen"]
X = ob[KEEP].copy()
X = pd.concat([X, pd.get_dummies(ob.index.dayofweek, prefix="dow").set_index(ob.index)], axis=1)
_b = X.select_dtypes(bool).columns
X[_b] = X[_b].astype(int)

import lightgbm as lgb

def v1_run(params, fold_list):
    """V1 파이프라인: 비율 타깃 + NaN 라벨 제거 + fold별 early stopping. (fold MAE 목록, best_iter 목록)"""
    ms, iters = [], []
    for f in fold_list:
        tr, va = f["train"], f["val"]
        ytr = np.log1p(y.loc[tr]) - np.log1p(r7.loc[tr])
        yva = np.log1p(y.loc[va]) - np.log1p(r7.loc[va])
        k = ytr.notna()
        m = lgb.LGBMRegressor(n_estimators=800, random_state=42, verbosity=-1, **params)
        m.fit(X.loc[tr][k], ytr[k], eval_set=[(X.loc[va], yva)],
              callbacks=[lgb.early_stopping(50, verbose=False)])
        p = np.expm1(pd.Series(m.predict(X.loc[va]), index=va) + np.log1p(r7.loc[va]))
        ms.append(float(np.mean(np.abs(y.loc[va] - p))))
        iters.append(m.best_iteration_ or 800)
    return ms, iters

SCREEN = dict(learning_rate=0.05, num_leaves=15, min_child_samples=10, subsample=0.9, colsample_bytree=0.9)
base_ms, _ = v1_run(SCREEN, SEL)
print(f"스크리닝 V1 (04 노트북 기준) 재현 완료 — 선택 fold 평균 MAE(원): {np.mean(base_ms):,.0f}")

In [ ]:
# §2 경량 튜닝 — Optuna TPE 60 trials, 목적함수 = 선택 fold 평균 MAE (test 미사용)
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(t):
    params = dict(
        learning_rate=t.suggest_float("learning_rate", 0.02, 0.1, log=True),
        num_leaves=t.suggest_int("num_leaves", 7, 31),
        min_child_samples=t.suggest_int("min_child_samples", 5, 30),
        subsample=t.suggest_float("subsample", 0.7, 1.0),
        colsample_bytree=t.suggest_float("colsample_bytree", 0.6, 1.0),
        reg_alpha=t.suggest_float("reg_alpha", 1e-8, 1.0, log=True),
        reg_lambda=t.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
    )
    return np.mean(v1_run(params, SEL)[0])

study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=60, show_progress_bar=False)
BEST = study.best_params
best_ms, best_iters = v1_run(BEST, SEL)

base_m, best_m = np.mean(base_ms), np.mean(best_ms)
improved = sum(b < s for s, b in zip(base_ms, best_ms))
cmp = pd.DataFrame({"스크리닝": [f"{v:,.0f}" for v in base_ms] + [f"{base_m:,.0f}"],
                    "튜닝(best)": [f"{v:,.0f}" for v in best_ms] + [f"{best_m:,.0f}"]},
                   index=[f["month"] for f in SEL] + ["평균"])
display(cmp)
print(f"개선 {(best_m/base_m-1)*100:+.1f}% | 개선 fold {improved}/5 | 채택 기준(-2%·과반): "
      f"{'✅ 충족 → 채택 (V1-t)' if best_m/base_m-1 <= -0.02 and improved >= 3 else '❌ 미달 → 스크리닝 유지'}")
print("채택 파라미터:", {k: round(v, 4) if isinstance(v, float) else v for k, v in BEST.items()})
print("fold별 best_iteration:", best_iters, "| 중앙값:", int(np.median(best_iters)))

### §2 관찰 — 튜닝 채택 판단

- 60 trials 결과 **-2.6%·3/5 fold 개선**으로 사전 기준을 충족 → **V1-t 채택**. 개선은 주로
  재개장 fold(2026-03, -7%대)에서 나왔고 나머지 fold는 동률 수준 — 보수화가 regime 구간의
  과반응을 줄인 효과로 해석.
- 파라미터 방향이 일관되게 **보수적**(트리 더 작게·느리게·피처/행 서브샘플 축소): 표본 112~240일
  규모에서 기대되는 최적 방향과 일치 — 우연한 선택-과적합만으로 보기 어려움.
- 그래도 60 trials가 선택 fold에 직접 맞춰진 만큼 **-2.6%를 액면 그대로 신뢰하지 않는다**:
  test 재개봉 없이 채택하되, 운영 재학습(주 단위)에서 MA-7 대비 skill을 상시 비교해 사후 검증.
  스크리닝 설정은 fallback 후보로 모델 카드에 병기.

In [ ]:
# §3 최종 모델 카드 — V1-t 정의 + 선택 구간 전체 적합의 gain 상위 (참고용, test 미사용)
card = pd.DataFrame([
    ("모델", "LightGBM 4.x 회귀 (lightgbm>=4.5, [ml] extra)"),
    ("타깃", "log1p(일 매출) − log1p(roll7_mean) — 최근 7영업일 평균 대비 편차"),
    ("복원", "ŷ = expm1(pred + log1p(roll7_mean))"),
    ("피처", "keep 20열 = 13열 + dow 원핫 7 (02_features §7, 03·04에서 hold 전부 기각)"),
    ("전처리", "preprocess.py 규칙 — 기상 보간·워밍업 NaN 라벨 제거·이상치 log-IQR k=3(개입 0)"),
    ("검증", "월 단위 walk-forward (val=영업일 ≥10일 월, 최종 fold test 봉인)"),
    ("지표", "MAE(주·선정 기준) + sMAPE(보고). MAPE 제외(소액일 왜곡)"),
    ("운영 목표", "naive-요일 대비 skill ≥ +15% · MA-7 우위 유지(미달 시 drift 경보)"),
    ("보조 baseline", "MA-7 (fallback·drift 감시)"),
    ("성능(상대)", "선택 fold: MA-7 대비 약 -11%(V1-t) · 봉인 test(스크리닝 V1): sMAPE 30.0%, MA-7 대비 -19.6%"),
    ("하이퍼파라미터", "V1-t = 위 §2 채택값 (fallback: 스크리닝 lr0.05/leaves15/mcs10/ss0.9/cs0.9)"),
], columns=["항목", "내용"]).set_index("항목")
display(card)

# 선택 구간 전체(train+val ≤ 2026-03-31)로 적합해 gain 상위 확인 — 서빙 모델 아님(Phase 7에서 재학습)
fit_idx = ob.index[ob.index <= SEL[-1]["val"].max()]
ytr_all = (np.log1p(y.loc[fit_idx]) - np.log1p(r7.loc[fit_idx]))
k = ytr_all.notna()
final = lgb.LGBMRegressor(n_estimators=int(np.median(best_iters)), random_state=42, verbosity=-1, **BEST)
final.fit(X.loc[fit_idx][k], ytr_all[k])
gain = pd.Series(final.booster_.feature_importance("gain"), index=X.columns)
gain = (gain / gain.sum() * 100).sort_values().tail(10)

fig, ax = plt.subplots(figsize=(7.5, 3.8), constrained_layout=True)
ax.barh(gain.index, gain.values, color=PAL["blue"], height=0.6)
ax.set_xlabel("gain 비중 (%)")
ax.set_title("V1-t 최종 적합(선택 구간)의 gain 상위 10 — M6.A7 SHAP 대상")
ax.grid(axis="y", visible=False)
plt.show()
print("MA-7 대비(선택 fold):", f"{best_m / 325457 - 1:+.1%}")

### §3 관찰 — 모델 카드

- gain 상위는 최근 운영 상태(roll7_atv·roll7_mean)·같은 요일 이력(lag_dow·roll4dow)·학사 주차·기온 순 —
  편차 모델에서 "평소 대비"를 흔드는 요인이 그대로 상위에 옴. **M6.A7 TreeSHAP은 이 모델에 적용**하면 기여도가 곧 "평소 대비 증감 요인"이라
  자연어 근거("내일은 목요일·기온 하락으로 평소보다 높음")와 1:1로 연결된다.
- 카드의 "성능(상대)"에서 test 수치는 **스크리닝 V1의 것** — V1-t는 test 미평가(봉인 원칙).
  운영 배포 후 첫 달 성적이 V1-t의 실질 test가 된다.

## §4 판정·다음 단계

**M6.A6 종료 판정** — 초기 모델(V1-t)·보조 baseline(MA-7)·평가 지표(MAE+sMAPE, MAPE 제외)·운영 목표
(상대 기준) 확정. 선정 사유·평가 보고서 = 04 노트북(비교) + 본 노트북(튜닝·카드). **산출물 충족.**

### spec 반영 (docs 브랜치 PR)

- `model_spec.md` §2: 베이스라인 비교 완료 주석(04 노트북 참조), 4·5단계는 M6.A9 판단으로 갱신
- §3: 초기 모델 확정 — LightGBM 비율 타깃 하이브리드 + 라이브러리 + 보조 MA-7 + 타깃(매장 일 매출) 명시
- §5: MVP 확정 피처 20열 참조(유동인구 1차 제외 주석)
- §7: CV를 "월 단위 walk-forward(ml_pipeline §6)"로 구체화 + 평가 지표·운영 목표 확정
- 말미 미확정 목록에서 "데이터 분리·평가 지표" 제거
- plan의 "feature_spec §5.2 ROI 갱신" 항목은 현행 문서와 불일치(§5.2=조회, ROI=§8.2 [2단계]) — 갱신 불요 보고

### 다음 마일스톤

- **M6.A7 XAI** — TreeSHAP(`shap.TreeExplainer`)를 V1-t에 적용, top-3 기여 피처 + 자연어 변환 prototype
  (shap 패키지 [ml] 추가 필요)
- **M6.A8 신뢰도 기준** — fold sMAPE 분포(30~64%) + 예측 구간(잔차 분위수) 기반 임계값 산정
- 담당자 확인: ① 타깃 간극(매장 일 매출 ↔ spec 메뉴별 1~3일) Phase 7 분해 설계 ② 검수 3건(변동 없음)